<table style="width: 100%; border-collapse: collapse; border: none; background: #f8fafc; border-left: 6px solid #1e3a8a; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #0f172a; font-size: 2.1em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        Taller Hands-On: Telemetría de Aire a Escala — Resumir, Aprender por Tandas y Contar la Historia 🌬️
      </h1>
      <p style="margin: 6px 0 0 0; color: #1e3a8a; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Big Data
      </p>
      <p style="margin: 4px 0 0 0; color: #64748b; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #1e3a8a; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        Taller Práctico Evaluativo · Módulo 03
      </span><br>
      <span style="color: #64748b; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #2563eb; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Big%20Data/homeworks/03_Analitica_Visualizacion_Hands_On.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## Objetivos de Aprendizaje 🔎

Este taller consolida el **Módulo 03: Analítica y Visualización en Big Data** ([README del módulo](../03%20-%20Analitica%20y%20Visualizacion%20en%20Big%20Data/README.md)) con un problema **nuevo**: la red de monitoreo de calidad del aire **AireClaro** (ficticia) genera lecturas y eventos que **no caben en memoria** ni en un gráfico. Tendrás que resumirlos, aprender de ellos por tandas, repartir el cómputo y contar el hallazgo con honestidad.

Al terminar serás capaz de:

1. **Resumir un flujo en una sola pasada** con un *reservorio* uniforme y un *Count-Min Sketch*, y **verificar con datos** sus cotas de error.
2. **Entrenar fuera de memoria** con `partial_fit` por trozos y detectar la **trampa del orden** de los datos.
3. **Demostrar** que el gradiente combinado de varios trabajadores (promedio **ponderado**) es el gradiente del lote completo, y que promediar sin pesar **no** lo es.
4. **Agregar antes de graficar** (*binning* + agrupación) y construir una figura **explicativa** cuyo título, anotaciones y ejes se **auditan por código** (propiedades de la figura, no píxeles).

> 📚 **Repasa si lo necesitas:** [Cuaderno 00: muestreo, reservorio y Count-Min](../03%20-%20Analitica%20y%20Visualizacion%20en%20Big%20Data/00_Introduccion_Cap2_y_Resumen.ipynb) · [Cuaderno 01: `partial_fit` y paralelismo de datos](../03%20-%20Analitica%20y%20Visualizacion%20en%20Big%20Data/01_Tecnicas_de_Analitica_ML_DL_NLP.ipynb) · [Cuaderno 02: agregar antes de graficar](../03%20-%20Analitica%20y%20Visualizacion%20en%20Big%20Data/02_Herramientas_de_Visualizacion_Tableau_PowerBI_D3.ipynb) · [Cuaderno 03: *storytelling*](../03%20-%20Analitica%20y%20Visualizacion%20en%20Big%20Data/03_Storytelling_con_Datos.ipynb).

---
## Instrucciones 📋

1. **Resuelve de forma autónoma:** en cada reto completa la celda de código marcada con `# TODO:`. Este cuaderno **no trae soluciones**.
2. **Autoverificación:** debajo de cada reto hay una celda con `assert`. Si tu solución es correcta imprime ✅; si no, el mensaje te dice qué revisar. No la modifiques.
3. **Puntaje:** cada reto indica su puntaje sugerido; el total del taller suma **100 puntos**.
4. Antes de entregar usa *Restart & Run All*: el cuaderno debe correr de principio a fin, sin errores y **sin `# TODO` pendientes**. Tarda menos de un minuto.
5. Todo funciona **sin internet y sin archivos externos** (datos sintéticos con semilla fija). Bibliotecas: `numpy`, `pandas`, `scikit-learn` y `matplotlib`. **No se usa *Spark* ni *PyTorch***: los mecanismos (sketches, trozos, gradientes por trabajador) los implementas tú.
6. Los datos son **inventados**: los resultados describen el mecanismo, no el desempeño de una red de monitoreo real.

In [ ]:
# ============================================================
# Preparación (no modificar): imports y datos sintéticos
# ============================================================
import math
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.text import Annotation
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
SEMILLA = 33

# ---- Reto 1: flujo de eventos (se genera por TROZOS; nunca existe completo en memoria) ----
N_SENSORES, N_EVENTOS, TAM_TROZO = 50_000, 300_000, 10_000
P_PRIMO = 2_147_483_647                      # primo 2^31 - 1, para el hash universal


def flujo_eventos():
    """Generador de trozos (ids_sensor, latencia_ms). Cada llamada reproduce el MISMO flujo."""
    rng = np.random.default_rng(SEMILLA)
    p = 1.0 / np.arange(1, N_SENSORES + 1) ** 1.1          # frecuencias tipo Zipf: pocos sensores muy activos
    p /= p.sum()
    for _ in range(N_EVENTOS // TAM_TROZO):
        yield rng.choice(N_SENSORES, size=TAM_TROZO, p=p), rng.lognormal(3.0, 0.6, TAM_TROZO)


# ---- Reto 2: mantenimiento de bombas (falla = 1); 8 variables en escalas muy distintas ----
rng2 = np.random.default_rng(SEMILLA + 10)
_escala = np.array([1.0, 5.0, 0.2, 10.0, 1.0, 3.0, 1.0, 2.0])
_centro = np.array([0.0, 20.0, 1.0, -50.0, 3.0, 0.0, 10.0, 5.0])
_peso = np.array([1.2, -0.8, 0.6, 0.0, 0.9, -0.5, 0.0, 0.4])
_Z = rng2.normal(size=(60_000, 8))
_Xb = _Z * _escala + _centro
_yb = rng2.binomial(1, 1 / (1 + np.exp(-(_Z @ _peso - 0.8))))
X_tr, y_tr, X_te, y_te = _Xb[:45_000], _yb[:45_000], _Xb[45_000:], _yb[45_000:]
TAM_B = 5_000                                # filas por trozo
orden_original = np.arange(len(X_tr))                        # el «archivo» llega como se generó
orden_por_clase = np.argsort(y_tr, kind="stable")            # el «archivo» llega ORDENADO por clase

# ---- Reto 3: clasificación logística con 4 variables + columna de unos (intercepto) ----
rng3 = np.random.default_rng(SEMILLA + 20)
N3 = 1003
X3 = np.column_stack([rng3.normal(size=(N3, 4)), np.ones(N3)])
y3 = rng3.binomial(1, 1 / (1 + np.exp(-(X3 @ np.array([1.0, -1.5, 0.7, 0.0, 0.3])))))
w_prueba = np.random.default_rng(SEMILLA + 3).normal(0, 0.5, 5)      # un punto cualquiera donde comparar gradientes

# ---- Reto 4: 240 000 lecturas de PM2.5 (µg/m³) de seis estaciones de AireClaro ----
ESTACIONES = {   # nombre: (lecturas, nivel base, hora del pico, amplitud del pico, ancho del pico en horas)
    "Tunja": (70_000, 14.0, 8, 6.0, 2.0), "Duitama": (50_000, 16.0, 18, 5.0, 2.0),
    "Paipa": (30_000, 10.0, 8, 3.0, 2.0), "Chiquinquirá": (40_000, 15.0, 17, 4.0, 2.0),
    "Villa de Leyva": (20_000, 8.0, 12, 1.0, 3.0), "Sogamoso": (30_000, 15.0, 22, 25.0, 1.5)}


def generar_lecturas():
    rng = np.random.default_rng(SEMILLA + 30)
    partes = []
    for nombre, (n, base, pico, amp, ancho) in ESTACIONES.items():
        minuto = rng.uniform(0, 1440, n)                       # minuto del día, continuo
        dif = np.abs(minuto / 60 - pico)
        dist = np.minimum(dif, 24 - dif)                       # distancia circular al pico (horas)
        pm25 = np.clip(rng.normal(base + amp * np.exp(-0.5 * (dist / ancho) ** 2), 4.0), 0, None)
        partes.append(pd.DataFrame({"estacion": nombre, "minuto": minuto, "pm25": pm25}))
    return pd.concat(partes, ignore_index=True).sample(frac=1, random_state=SEMILLA).reset_index(drop=True)


lecturas = generar_lecturas()

print("Flujo:", f"{N_EVENTOS:,} eventos en {N_EVENTOS // TAM_TROZO} trozos de {TAM_TROZO:,}",
      "| Bombas:", X_tr.shape, "(entrenamiento)", X_te.shape, "(prueba)",
      "| Logística:", X3.shape, "| Lecturas de aire:", lecturas.shape)

---
## Reto 1: Resumir un flujo que no cabe — reservorio y *Count-Min Sketch* con cota verificada 🌊

**Contexto:** AireClaro recibe un flujo de **300 000 eventos** `(id_sensor, latencia_ms)` entre 50 000 sensores, entregado en 30 trozos que no puedes guardar. En **una sola pasada** debes (a) conservar una **muestra uniforme** de $K$ latencias y (b) mantener un **sketch** que estime cuántos eventos produjo cada sensor, **sin** un contador por sensor.

**Enunciado**
1. Dimensiona el *Count-Min Sketch* para un error relativo $\varepsilon$ y una probabilidad de fallo $\delta$: ancho $w=\lceil e/\varepsilon\rceil$ y profundidad $d=\lceil \ln(1/\delta)\rceil$. Garantía: para todo sensor $x$, $f(x)\le \hat f(x)$ y $P\big(\hat f(x)-f(x) > \varepsilon N\big)\le \delta$, con $N$ el total de eventos y $\hat f(x)=\min_i T[i,\,h_i(x)]$.
2. Genera $d$ funciones de hash universales $h_i(x)=\big((a_i x+b_i)\bmod p\big)\bmod w$, con $p=2^{31}-1$ (`P_PRIMO`), $a_i\in[1,p)$ y $b_i\in[0,p)$ sorteados con `rng_hash`.
3. En **un solo recorrido** de `flujo_eventos()` actualiza: la `tabla` de contadores ($d\times w$; en cada fila suma 1 en la celda $h_i(x)$ de cada evento), el `reservorio` de $K$ latencias (Algoritmo R con `rng_res`: los primeros $K$ entran; el elemento número $n$ —contando desde 1— entra con probabilidad $K/n$ y reemplaza a uno al azar), `n_vistos` y la suma de **todas** las latencias.
4. `estimar(ids)`: devuelve $\hat f$ para un arreglo de ids. Calcula también `real` (conteos exactos con `np.bincount`: **solo para verificar**; un sistema real no puede permitírselo).
5. Compara: `frac_viol` = fracción de los 50 000 sensores con $\hat f(x)-f(x) > \varepsilon N$; y la media de latencia del flujo frente a la del reservorio, con la cota `cota_media` $=5\,s/\sqrt{K}$ ($s$: desviación estándar de la muestra).

**Criterios de aceptación:** cada fila de `tabla` suma $N$; el sketch **nunca subestima**; `frac_viol` $\le \delta$; la tabla ocupa menos de la cuarta parte de lo que ocuparía un contador por sensor; el reservorio tiene $K$ valores distintos **del flujo**, no sesgados hacia el inicio (entre 30 % y 70 % viene de la segunda mitad); y el error de la media queda dentro de `cota_media`.

**Puntaje sugerido:** 20 puntos.

In [ ]:
# TODO: completa
K = 400                                          # tamaño del reservorio
EPS, DELTA = 0.002, 0.01                         # error relativo y probabilidad de fallo deseados
rng_res = np.random.default_rng(SEMILLA + 1)     # para el reservorio
rng_hash = np.random.default_rng(SEMILLA + 2)    # para los coeficientes del hash

w = d = None            # ancho y profundidad del sketch (a partir de EPS y DELTA)
a = b = None            # d coeficientes aleatorios: a en [1, P_PRIMO), b en [0, P_PRIMO)
tabla = None            # contadores enteros (d x w), inician en 0
reservorio = None       # arreglo de K latencias
n_vistos = 0            # eventos procesados
suma_lat = 0.0          # suma de TODAS las latencias (acumulada; no las guardes)
real = None             # conteos exactos por sensor (solo para verificar)

# TODO: UN solo recorrido del flujo (actualiza tabla, reservorio, n_vistos, suma_lat y real)
for ids, lat in flujo_eventos():
    ...


def estimar(ids):
    # TODO: estimación Count-Min de la frecuencia de cada id: mínimo de las d celdas que le corresponden
    return None


media_real = None       # suma_lat / n_vistos
media_res = None        # media del reservorio
cota_media = None       # 5 * (desviación estándar del reservorio) / sqrt(K)
est = None              # estimar(np.arange(N_SENSORES))
frac_viol = None        # fracción de sensores con (est - real) > EPS * n_vistos

In [ ]:
# Autoverificación del Reto 1 (no modificar)
assert w == math.ceil(math.e / EPS) and d == math.ceil(math.log(1 / DELTA)), "Dimensiones: w = ceil(e / EPS) y d = ceil(ln(1 / DELTA))."
assert isinstance(tabla, np.ndarray) and tabla.shape == (d, w) and np.issubdtype(tabla.dtype, np.integer), "'tabla' debe ser un arreglo de enteros de forma (d, w)."
assert n_vistos == N_EVENTOS, "n_vistos debe contar los 300 000 eventos (recorre TODOS los trozos de flujo_eventos())."
assert (tabla.sum(axis=1) == N_EVENTOS).all(), "Cada evento suma 1 en UNA celda de CADA fila: todas las filas de la tabla deben sumar N_EVENTOS."
_ids_all, _lat_all = zip(*flujo_eventos())
_ids_all, _lat_all = np.concatenate(_ids_all), np.concatenate(_lat_all)
_real = np.bincount(_ids_all, minlength=N_SENSORES)
assert real is not None and np.array_equal(real, _real), "'real' debe ser el conteo exacto por sensor (np.bincount de los ids de todo el flujo)."
assert np.isclose(suma_lat, _lat_all.sum(), rtol=1e-9), "suma_lat debe acumular la suma de TODAS las latencias del flujo."
assert isinstance(reservorio, np.ndarray) and len(reservorio) == K and len(np.unique(reservorio)) == K, "El reservorio debe tener K latencias distintas."
assert np.isin(reservorio, _lat_all).all(), "Todos los valores del reservorio deben ser latencias que aparecieron en el flujo."
_mitad = np.isin(reservorio, _lat_all[N_EVENTOS // 2:]).mean()
assert 0.3 <= _mitad <= 0.7, f"Una muestra uniforme toma ≈ 50 % de la segunda mitad del flujo; la tuya {_mitad:.0%}. Revisa la probabilidad K / n de reemplazo (n cuenta desde 1) y que el reemplazo sea en una posición al azar."
assert abs(media_real - _lat_all.mean()) < 1e-6 and abs(media_res - reservorio.mean()) < 1e-9, "media_real = suma_lat / n_vistos y media_res = reservorio.mean()."
assert np.isclose(cota_media, 5 * reservorio.std(ddof=1) / math.sqrt(K), rtol=0.01), "cota_media = 5 * s / sqrt(K), con s la desviación estándar del reservorio."
assert abs(media_res - media_real) <= cota_media, "El error de la media estimada con el reservorio debería caber en 5 errores estándar: revisa que la muestra sea uniforme."
_e = np.asarray(estimar(np.arange(N_SENSORES)))
assert _e.shape == (N_SENSORES,) and np.array_equal(_e, np.asarray(est)), "'est' debe ser estimar(np.arange(N_SENSORES)), un arreglo con un valor por sensor."
assert (est >= _real).all(), "Un Count-Min Sketch NUNCA subestima (los contadores solo suman): revisa que uses el mínimo de las d filas y que cada fila sume su propia celda."
_viol = float(np.mean(est - _real > EPS * N_EVENTOS))
assert abs(frac_viol - _viol) < 1e-12, "frac_viol es la fracción de sensores con (est - real) > EPS * n_vistos."
assert frac_viol <= DELTA, "La cota dice que a lo sumo una fracción DELTA de los sensores puede pasarse de EPS * N: revisa las funciones de hash (a, b distintos por fila) y las dimensiones."
assert tabla.size < N_SENSORES / 4, "El sketch debe ocupar mucho menos que un contador por sensor."
print(f"✅ Reto 1 superado: sketch {d}×{w} = {tabla.size:,} celdas ({tabla.size / N_SENSORES:.0%} de un contador por sensor) | error máx. = {(est - real).max()} "
      f"(cota EPS·N = {EPS * N_EVENTOS:.0f}) | violaciones = {frac_viol:.2%} | media: {media_res:.2f} vs {media_real:.2f} ms (cota ±{cota_media:.2f})")

---
## Reto 2: Aprender fuera de memoria con `partial_fit` y la trampa del orden 🧱

**Contexto:** el historial de 45 000 inspecciones de bombas (8 variables, falla = 1) vive «en disco» y solo puedes cargar **trozos de 5 000 filas**. Vas a entrenar un clasificador logístico por trozos y a comprobar que **el orden en que llega el archivo importa**.

**Enunciado**
1. Implementa `entrenar_por_trozos(X, y, orden, epocas=5)`: (a) `esc = StandardScaler()` con **una** pasada de `partial_fit` sobre los trozos (un trozo son `TAM_B` posiciones consecutivas de `orden`); (b) `clf = SGDClassifier(loss="log_loss", random_state=0)` entrenado con `partial_fit(esc.transform(trozo), y_trozo, classes=[0, 1])` en `epocas` pasadas sobre los trozos. Devuelve `(esc, clf)`.
2. Referencia **en memoria**: `StandardScaler` + `LogisticRegression(max_iter=1000)` ajustados con todo `X_tr`; `acc_ref` es su exactitud en `X_te`.
3. `acc_trozos`: exactitud en `X_te` de lo que devuelve `entrenar_por_trozos` con `orden_original`.
4. `acc_por_clase`: lo mismo con `orden_por_clase` (el archivo llega ordenado: primero todas las filas de clase 0 y luego las de clase 1).

**Criterios de aceptación:** el escalador incremental coincide con media y desviación globales; `acc_trozos` queda a menos de 0.03 de `acc_ref`; y con el archivo ordenado por clase la exactitud **cae al menos 0.10** (el modelo «olvida» lo anterior).

**Puntaje sugerido:** 20 puntos.

In [ ]:
def entrenar_por_trozos(X, y, orden, epocas=5):
    # TODO: 1) esc = StandardScaler(); UNA pasada de esc.partial_fit(X[idx]) con idx = orden[i:i + TAM_B]
    #       2) clf = SGDClassifier(loss="log_loss", random_state=0); `epocas` pasadas con
    #          clf.partial_fit(esc.transform(X[idx]), y[idx], classes=[0, 1])
    esc = clf = None
    return esc, clf


# TODO: completa
acc_ref = None          # exactitud en X_te de StandardScaler + LogisticRegression(max_iter=1000) ajustados con TODO X_tr
acc_trozos = None       # exactitud en X_te con entrenar_por_trozos(..., orden_original)
acc_por_clase = None    # exactitud en X_te con entrenar_por_trozos(..., orden_por_clase)

In [ ]:
# Autoverificación del Reto 2 (no modificar)
_e, _c = entrenar_por_trozos(X_tr, y_tr, orden_original)
assert isinstance(_e, StandardScaler) and isinstance(_c, SGDClassifier) and hasattr(_c, "coef_"), "entrenar_por_trozos debe devolver (StandardScaler, SGDClassifier) ya ajustados."
assert _e.n_samples_seen_ == len(X_tr), "El escalador debe ver cada fila UNA sola vez (una pasada de partial_fit sobre los trozos, no una por época)."
assert np.allclose(_e.mean_, X_tr.mean(axis=0)) and np.allclose(_e.scale_, X_tr.std(axis=0), rtol=1e-6), "La media y la escala acumuladas por trozos deben coincidir con las globales."
_ref_esc = StandardScaler().fit(X_tr)
_ref = LogisticRegression(max_iter=1000).fit(_ref_esc.transform(X_tr), y_tr)
assert acc_ref is not None and abs(acc_ref - _ref.score(_ref_esc.transform(X_te), y_te)) < 1e-9, "acc_ref: exactitud en X_te de la referencia en memoria (escalador + LogisticRegression(max_iter=1000), ajustados con TODO X_tr)."
assert acc_trozos is not None and abs(acc_trozos - _c.score(_e.transform(X_te), y_te)) < 1e-9, "acc_trozos: exactitud en X_te de lo que devuelve entrenar_por_trozos(X_tr, y_tr, orden_original), con epocas=5 y random_state=0."
assert acc_trozos >= acc_ref - 0.03, f"Por trozos deberías acercarte a la referencia en memoria (diferencia < 0.03): {acc_trozos:.3f} vs {acc_ref:.3f}. ¿Escalaste cada trozo con esc.transform y pasaste classes=[0, 1]?"
_e2, _c2 = entrenar_por_trozos(X_tr, y_tr, orden_por_clase)
assert acc_por_clase is not None and abs(acc_por_clase - _c2.score(_e2.transform(X_te), y_te)) < 1e-9, "acc_por_clase: exactitud en X_te con entrenar_por_trozos(X_tr, y_tr, orden_por_clase)."
assert acc_trozos - acc_por_clase >= 0.10, "Con el archivo ordenado por clase la exactitud debería caer al menos 0.10: ¿estás recorriendo los trozos en el orden dado por `orden`?"
print(f"✅ Reto 2 superado: en memoria {acc_ref:.3f} | por trozos {acc_trozos:.3f} | archivo ordenado por clase {acc_por_clase:.3f} (caída de {acc_trozos - acc_por_clase:.3f})")

---
## Reto 3: SGD con paralelismo de datos — el gradiente promedio *es* el del lote completo ⚙️

**Contexto:** para entrenar más rápido, $K$ trabajadores reciben cada uno un **fragmento** de los datos, calculan su gradiente local y un coordinador los combina antes de actualizar los pesos (el esquema síncrono de los cuadernos del módulo). Vas a **demostrarlo** con regresión logística en NumPy, incluso con fragmentos de **tamaños distintos**.

**Enunciado**
1. `gradiente(w, X, y)`: gradiente de la pérdida logística media, $\nabla L(w)=\dfrac1n X^\top\big(\sigma(Xw)-y\big)$ con $\sigma(z)=1/(1+e^{-z})$.
2. `gradiente_paralelo(w, X, y, tamanos)`: reparte las filas en fragmentos consecutivos de esos tamaños (`np.split` con `np.cumsum`), calcula el gradiente local de cada uno y los combina con el promedio **ponderado** $g=\sum_k \frac{n_k}{n}\,g_k$.
3. Con `tam_desiguales = [100, 300, 603]` y `w_prueba`: `g_completo`, `g_pond` (tu `gradiente_paralelo`) y `g_ingenuo` (promedio **sin** ponderar de los gradientes locales); `dif_pond` y `dif_ingenuo` son la máxima diferencia absoluta de cada uno respecto de `g_completo`.
4. `entrenar(tamanos, pasos=300, lr=1.0)`: descenso de gradiente síncrono desde $w=0$ con `gradiente_paralelo`; devuelve `(w, historial)` con la pérdida inicial y la de cada paso (`pasos + 1` valores). Entrena con 1 trabajador (`w_k1`, `hist_k1`) y con 4 trabajadores de tamaños `[250, 250, 250, 253]` (`w_k4`, `hist_k4`).

**Criterios de aceptación:** tu `gradiente` coincide con **diferencias finitas**; `dif_pond` es ≈ 0 (error de redondeo) y `dif_ingenuo` es claramente mayor; las trayectorias de 1 y 4 trabajadores son **idénticas**; la pérdida baja en cada paso y termina cerca del óptimo.

**Puntaje sugerido:** 20 puntos.

In [ ]:
def perdida(w, X, y):
    """Pérdida logística media (dada)."""
    z = X @ w
    return float(np.mean(np.logaddexp(0.0, z) - y * z))


def gradiente(w, X, y):
    # TODO: (1 / n) * X^T (sigmoide(X w) - y)
    return None


def gradiente_paralelo(w, X, y, tamanos):
    # TODO: fragmentos consecutivos de esos tamaños -> gradiente local de cada uno -> promedio PONDERADO por n_k / n
    return None


def entrenar(tamanos, pasos=300, lr=1.0):
    # TODO: w = 0; en cada paso w <- w - lr * gradiente_paralelo(...); historial = pérdida inicial + pérdida tras cada paso
    return None, None


# TODO: completa
tam_desiguales = [100, 300, 603]
g_completo = None               # gradiente del lote completo en w_prueba
g_pond = None                   # gradiente_paralelo(w_prueba, X3, y3, tam_desiguales)
g_ingenuo = None                # promedio SIN ponderar de los gradientes locales de esos mismos fragmentos
dif_pond = dif_ingenuo = None   # np.abs(... - g_completo).max()

w_k1 = hist_k1 = None           # entrenar([N3])
w_k4 = hist_k4 = None           # entrenar([250, 250, 250, 253])

In [ ]:
# Autoverificación del Reto 3 (no modificar)
_w = np.random.default_rng(0).normal(size=5)
_g = gradiente(_w, X3, y3)
assert _g is not None and np.shape(_g) == (5,), "gradiente debe devolver un vector de 5 componentes."
_num = np.array([(perdida(_w + h, X3, y3) - perdida(_w - h, X3, y3)) / 2e-6 for h in 1e-6 * np.eye(5)])
assert np.allclose(_g, _num, atol=1e-6), "Tu gradiente no coincide con las diferencias finitas de la pérdida: revisa la fórmula (1/n) X^T (sigmoide(Xw) - y)."
_n = len(y3)
for _tam in ([N3], [1, N3 - 1], [100, 300, 603], [250, 250, 250, 253]):
    _gp = gradiente_paralelo(_w, X3, y3, _tam)
    assert _gp is not None and np.allclose(_gp, _g, atol=1e-12), f"gradiente_paralelo con tamaños {_tam} debe dar el gradiente del lote completo: pondera cada gradiente local por n_k / n."
assert np.allclose(g_completo, gradiente(w_prueba, X3, y3), atol=1e-12), "g_completo es el gradiente de TODO el lote en w_prueba."
assert np.allclose(g_pond, g_completo, atol=1e-12), "g_pond debe salir de gradiente_paralelo(w_prueba, X3, y3, tam_desiguales)."
assert dif_pond is not None and dif_pond < 1e-12, "dif_pond debe ser ≈ 0: np.abs(g_pond - g_completo).max()."
assert dif_ingenuo is not None and dif_ingenuo > 1e-3 and abs(dif_ingenuo - np.abs(g_ingenuo - g_completo).max()) < 1e-12, "El promedio SIN ponderar (np.mean de los gradientes locales) NO es el gradiente del lote cuando los fragmentos son desiguales; dif_ingenuo debe medirlo."
for _h in (hist_k1, hist_k4):
    assert _h is not None and len(_h) == 301, "El historial debe tener pasos + 1 = 301 valores (pérdida inicial y la de cada paso)."
    assert np.all(np.diff(_h) <= 1e-12), "La pérdida debe bajar (o quedar igual) en cada paso."
assert _h[0] > _h[-1] + 0.1, "La pérdida debe bajar de forma apreciable respecto de la inicial."
assert np.allclose(w_k1, w_k4, atol=1e-9) and np.allclose(hist_k1, hist_k4, atol=1e-9), "Con el promedio ponderado, 1 y 4 trabajadores deben recorrer EXACTAMENTE la misma trayectoria."
_opt = LogisticRegression(C=1e8, fit_intercept=False, max_iter=5000).fit(X3, y3).coef_[0]
assert perdida(w_k1, X3, y3) < perdida(_opt, X3, y3) + 0.01, "Tras 300 pasos la pérdida debe quedar a menos de 0.01 del óptimo."
print(f"✅ Reto 3 superado: diferencia ponderada = {dif_pond:.1e} | sin ponderar = {dif_ingenuo:.3f} | "
      f"pérdida {hist_k1[0]:.3f} → {hist_k1[-1]:.3f} | 1 vs 4 trabajadores: {np.abs(w_k1 - w_k4).max():.1e}")

---
## Reto 4: Agregar antes de graficar y contar la historia en una sola figura 🎯

**Contexto:** la dirección de AireClaro pregunta en qué estación y a qué hora se concentra el problema. Dibujar las 240 000 lecturas sería ilegible (y lento); la salida es **agregar** y luego **diseñar la figura para explicar**, no para explorar. La calidad de la figura se auditará **por código**, sobre propiedades del objeto `Figure`.

**Enunciado**
1. *Binning:* añade `lecturas["hora"]`, la hora entera (0-23) a partir de `minuto`.
2. `agg`: `DataFrame` con **144 filas** y columnas `estacion`, `hora`, `media` (PM2.5 medio) y `n` (número de lecturas) por estación y hora; `reduccion` = lecturas crudas / filas de `agg`.
3. `media_global`: PM2.5 medio de **todas** las lecturas calculado **solo desde `agg`** (ponderando por `n`); `media_ingenua`: la media simple de `agg["media"]`.
4. `foco`, `hora_pico` y `pico`: la estación con el promedio horario más alto, su hora y su valor.
5. Figura (`fig`, `ax`): una línea por estación con los datos de `agg` —las cinco «normales» con trazo **fino** y la `foco` con trazo **grueso** y color distinto—; **título** que diga la conclusión y nombre la estación; ejes **rotulados** (con unidades); eje $y$ que **empiece en 0**; y **una anotación** (`ax.annotate`, con flecha) sobre el pico, cuyo texto incluya su valor.

**Criterios de aceptación:** `agg` y las medias coinciden con las calculadas directamente sobre los datos crudos; `media_ingenua` difiere de `media_global` (por qué: ponderar importa); la figura dibuja 6 series de 24 puntos (nada de 240 000 puntos), exactamente **una** destacada; título con ≥ 5 palabras que nombra al `foco`; ejes con etiqueta; entre 1 y 3 anotaciones, una apuntando a `(hora_pico, pico)` con un número en su texto; y $y$ desde 0.

**Puntaje sugerido:** 25 puntos.

In [ ]:
# TODO: completa
lecturas["hora"] = None     # hora entera 0-23 a partir de "minuto" (binning)
agg = None                  # DataFrame: estacion, hora, media (PM2.5 medio), n (nº de lecturas) -> 144 filas
reduccion = None            # len(lecturas) / len(agg)
media_global = None         # media de TODAS las lecturas, calculada SOLO desde agg (ponderada por n)
media_ingenua = None        # media simple de agg["media"] (sin pesar)
foco = hora_pico = pico = None     # estación, hora y valor del promedio horario más alto

fig, ax = plt.subplots(figsize=(9, 5))
# TODO: 1) una línea por estación con datos de agg: trazo fino para las 5 normales, grueso y de otro color para `foco`
#       2) ax.set_title(...) con la conclusión (nombra la estación y la hora)
#       3) ax.set_xlabel(...) y ax.set_ylabel(...) con unidades
#       4) ax.set_ylim(bottom=0)
#       5) ax.annotate("... valor ...", xy=(hora_pico, pico), xytext=(..., ...), arrowprops=dict(arrowstyle="->"))
plt.show()

In [ ]:
# Autoverificación del Reto 4 (no modificar)
_h = (lecturas["minuto"] // 60).astype(int)
assert lecturas["hora"].notna().all() and np.array_equal(lecturas["hora"].to_numpy(), _h.to_numpy()), "lecturas['hora'] debe ser la hora entera: (minuto // 60).astype(int)."
assert isinstance(agg, pd.DataFrame) and list(agg.columns) == ["estacion", "hora", "media", "n"], "agg debe tener las columnas ['estacion', 'hora', 'media', 'n'], en ese orden."
assert len(agg) == 144 and int(agg["n"].sum()) == len(lecturas), "agg debe tener 6 estaciones × 24 horas = 144 filas cuyo 'n' sume el total de lecturas."
_ref = lecturas.groupby(["estacion", "hora"])["pm25"].agg(["mean", "size"])
_m = agg.merge(_ref.reset_index(), on=["estacion", "hora"])
assert len(_m) == 144 and np.allclose(_m["media"], _m["mean"]) and (_m["n"] == _m["size"]).all(), "La media y el conteo por (estación, hora) no coinciden con los datos crudos."
assert reduccion is not None and abs(reduccion - len(lecturas) / len(agg)) < 1e-9, "reduccion = len(lecturas) / len(agg)."
assert abs(media_global - lecturas["pm25"].mean()) < 1e-9, "media_global debe reproducir la media de las lecturas crudas usando solo agg: sum(media * n) / sum(n)."
assert abs(media_ingenua - agg["media"].mean()) < 1e-12 and abs(media_ingenua - media_global) > 0.05, "media_ingenua es agg['media'].mean() (sin pesar) y debe diferir de media_global: las celdas tienen distinto número de lecturas."
_f = agg.loc[agg["media"].idxmax()]
assert foco == _f["estacion"] and hora_pico == int(_f["hora"]) and abs(pico - _f["media"]) < 1e-9, "foco, hora_pico y pico salen de la fila de agg con la media más alta."
# ---- La figura (propiedades del objeto, no píxeles) ----
assert isinstance(fig, plt.Figure) and ax in fig.axes, "Usa `fig, ax = plt.subplots(...)` y dibuja en `ax`."
_series = [l for l in ax.lines if len(l.get_xdata()) == 24]
assert len(_series) == 6, "Dibuja 6 series (una por estación) de 24 puntos cada una, con los datos AGREGADOS."
assert sum(len(l.get_xdata()) for l in ax.lines) + sum(len(c.get_offsets()) for c in ax.collections) <= len(agg), "No dibujes las lecturas crudas: solo los puntos de agg (144)."
_lw = [l.get_linewidth() for l in _series]
assert max(_lw) > min(_lw) and _lw.count(max(_lw)) == 1, "Exactamente UNA serie debe tener el trazo más grueso (la del foco); las demás, finas."
_dest = [l for l in _series if l.get_linewidth() == max(_lw)][0]
assert abs(np.max(_dest.get_ydata()) - pico) < 1e-6, "La serie destacada debe ser la de la estación foco."
assert len({l.get_color() for l in _series if l.get_linewidth() != max(_lw)} | {_dest.get_color()}) >= 2, "La serie destacada debe tener un color distinto del de las demás."
_t = ax.get_title()
assert _t.strip() and len(_t.split()) >= 5 and foco in _t, "El título debe tener al menos 5 palabras, decir la conclusión y nombrar la estación foco."
assert ax.get_xlabel().strip() and ax.get_ylabel().strip(), "Rotula ambos ejes (con unidades)."
assert ax.get_ylim()[0] <= 0, "El eje y debe empezar en 0 (ax.set_ylim(bottom=0)) para no exagerar las diferencias."
_an = [t for t in ax.texts if isinstance(t, Annotation)]
assert 1 <= len(_an) <= 3, "Usa entre 1 y 3 anotaciones (ax.annotate): lo importante, no todo."
assert any(abs(t.xy[0] - hora_pico) <= 0.5 and abs(t.xy[1] - pico) <= 0.5 and any(c.isdigit() for c in t.get_text()) for t in _an), "Una anotación debe apuntar a (hora_pico, pico) y su texto debe incluir el valor numérico."
print(f"✅ Reto 4 superado: {len(lecturas):,} lecturas → {len(agg)} puntos (×{reduccion:,.0f} menos) | media global {media_global:.2f} vs ingenua {media_ingenua:.2f} | foco: {foco} a las {hora_pico} h ({pico:.1f})")

---
## Reto 5: Conclusiones 📝

**Puntaje sugerido:** 15 puntos (sin código). Escribe **5 a 8 líneas** en la celda de abajo interpretando *tus* resultados. Debe responder:

- ¿Qué garantiza y qué **no** garantiza el *Count-Min Sketch* (usa tus cifras: error máximo frente a $\varepsilon N$, `frac_viol`, memoria)? ¿Por qué el reservorio, y no «los primeros $K$ eventos», sirve para estimar la media?
- ¿Qué enseña la caída de `acc_por_clase` sobre `partial_fit` y el orden del archivo? ¿Qué harías en un sistema real para evitarla?
- ¿Por qué hay que **ponderar** al combinar gradientes (o medias de grupos), y qué cambia si los trabajadores tienen fragmentos iguales o desiguales? ¿Qué **no** te dice la equivalencia sobre el tiempo de entrenamiento?
- Tu figura pasó la auditoría. ¿Qué **no** puede verificar un `assert` (que el título sea verdad, que la agregación por hora no oculte algo) y qué revisión humana añadirías?

---
#### ✍️ Tus conclusiones

_Escribe aquí tus 5-8 líneas. Reemplaza este texto por tu análisis._

---
### 🧠 Rúbrica de evaluación

| Criterio | Puntos |
|---|---|
| Reto 1: reservorio y *Count-Min Sketch* en una pasada, con cotas verificadas | 20 |
| Reto 2: entrenamiento por trozos con `partial_fit` y trampa del orden | 20 |
| Reto 3: gradiente por trabajador ponderado = gradiente del lote completo | 20 |
| Reto 4: agregación antes de graficar y figura explicativa auditada | 25 |
| Reto 5: conclusiones argumentadas con tus cifras (5-8 líneas) | 15 |
| **Total** | **100** |

### ✅ Checklist de entrega

- [ ] Reinicié el kernel y ejecuté *Run All* sin errores.
- [ ] No quedan `# TODO` ni variables en `None`.
- [ ] Las cuatro celdas de autoverificación imprimen ✅.
- [ ] El flujo se recorrió en una sola pasada y los datos por trozos nunca se cargaron completos.
- [ ] La figura dibuja datos agregados, con título-conclusión, ejes rotulados, eje $y$ desde 0 y pocas anotaciones.
- [ ] Escribí mis conclusiones con cifras de mis propios resultados.

---
<div align="center">
  <p style="font-size: 0.9em; color: #64748b;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Asignatura: Big Data</i>
  </p>
</div>